# PHASE 1 — MNIST, Gaussian Naïve Bayes và Logistic Regression

> **Đồ án thực hành giữa kỳ — Nhận dạng chữ số viết tay MNIST**  
> **Vai trò:** Dữ liệu dùng chung + 2/4 mô hình bắt buộc  
> **Mô hình trong phase:** GaussianNB-NP; LogReg-GD-NP  
> **Tình trạng:** *Bộ khung giao việc; toàn bộ notebook chỉ có ô Markdown TODO, chưa có code hoặc số liệu.*

**Tài liệu bắt buộc:** `DoAn_ThucHanh_NhanDang.pdf` (Đồ án 1, trang 1–2; phụ lục trang 4).  
**Tham khảo:** `BTVN_Lab01.ipynb`, `NaiveBayes(1).ipynb`, `TL_Lab03.ipynb`.  
**Kế hoạch tổng:** `plan.md`.

**Luồng áp dụng:** **DATA → X, y → split → preprocessing → fit → predict → metrics → visualize → analyze** (theo `BTVN_Lab01.ipynb`).

---

## 0. Mục tiêu và chuẩn bị

**Bảng nhiệm vụ theo format 6 bước của `NaiveBayes(1).ipynb`:**

| Bước | Nội dung | Đầu ra tối thiểu |
|---|---|---|
| 1 | Đọc và mô tả MNIST | Kích thước, nhãn, shape, dtype |
| 2 | Kiểm tra chất lượng và chia tập | Missing, duplicate, train/val/test cố định |
| 3 | EDA và tiền xử lý | ≥3 biểu đồ, [0,1], (N,784) |
| 4 | Tự cài NB + Logistic GD | Công thức, class/fit/predict, kiểm tra |
| 5 | Thí nghiệm siêu tham số | ≥1 biểu đồ learning rate/accuracy/F1 |
| 6 | Đánh giá & biện luận | **Accuracy, precision/recall/F1 macro, CM cho cả 3 split** |

**TODO:** Thống nhất mã model `GaussianNB-NP`, `LogReg-GD-NP`, `SEED=42`, chuẩn CSV theo `plan.md`. Code sklearn **chỉ đối chiếu**, không huấn luyện lõi bằng sklearn.

### 0.1. Môi trường, thư viện và đường dẫn

**TODO:**
1. Import NumPy, pandas, Matplotlib, thư viện tải MNIST, `time.perf_counter`, công cụ chia stratified và metric.
2. Thiết lập `SEED=42`, `labels=0..9`, đường dẫn `artifacts/`, `figures/`, `results/`, `models/`; thống nhất tên biến `X_train`, `X_val`, `X_test`, `y_train`, ...
3. Ghi phiên bản Python và các thư viện; đánh dấu các import sklearn nào chỉ là đối chiếu/phụ trợ.

**Tái sử dụng:** `TL_Lab03.ipynb` phần `0` / ô import, seed, thời gian và plotting; `BTVN_Lab01.ipynb` phần quy định chung.

## 1. MNIST — Tải, mô tả, EDA và chia tập

### 1.1. Đọc & mô tả dữ liệu

**TODO:**
1. Tải **MNIST chuẩn 28×28**, có **60.000** ảnh train gốc và **10.000** ảnh test gốc; không dùng `sklearn.load_digits` (8×8).
2. In `shape`, `dtype`, nhãn unique, min/max pixel, số lượng feature sau flatten.
3. Hiển thị vài ảnh 0–9 và đối chiếu nhãn.

**Sản phẩm:** Bảng mô tả dữ liệu + ảnh mẫu có nhãn.

### 1.2. Kiểm tra chất lượng dữ liệu

**TODO:**
1. Kiểm tra NaN/Inf, dtypes, nhãn ngoài 0..9, giá trị pixel âm/ngoài khoảng nguồn; thống kê ảnh trùng và pixel hằng số (cân nhắc chi phí tính toán).
2. Đếm số ảnh theo từng nhãn trên dữ liệu train gốc và test gốc.
3. Ghi nhận ảnh trùng nếu có; **không tự ý xóa** làm thay đổi giao thức so sánh và báo cáo rõ quyết định làm sạch.

**Tái sử dụng:** Checklist missing/duplicate/constant/range trong `NaiveBayes(1).ipynb` mục 2.1.2 / 2.2.2 và `BTVN_Lab01.ipynb` phần bài tập.

### 1.3. EDA & trực quan hoá

**TODO — tối thiểu 3 biểu đồ có tiêu đề/nhãn trục/legend khi cần:**
1. Lưới ảnh mẫu 0–9 theo nhãn (ảnh **28×28**).
2. Biểu đồ số lượng/tỉ lệ 10 lớp trên **train / validation / test** sau chia.
3. Ảnh trung bình (mean image) của từng lớp 0–9 chỉ từ **tập train**; có thể bổ sung histogram giá trị pixel hoặc một heatmap.

**Tái sử dụng:** `NaiveBayes(1).ipynb` mục 2.2.3 (đổi ảnh 8×8 thành 28×28); `BTVN_Lab01.ipynb` nguyên tắc ≥3 biểu đồ có nhận xét.

### 1.4. Nhận xét về dữ liệu

**TODO — trả lời bằng số liệu/ảnh:**
- Các lớp 0–9 có cân bằng không? Vì sao cần giữ phân tầng khi chia train/validation?
- Chữ số nào có hình dạng trung bình tương tự nhau; dự kiến cặp nhãn dễ nhầm?
- Pixel có tương quan không gian thế nào, và điều đó ảnh hưởng giả định độc lập của GaussianNB ra sao?
- Dữ liệu ảnh có khác dữ liệu bảng `Breast Cancer`/`Digits` từ notebook Naive Bayes ở chỗ nào?

### 1.5. Tạo train / validation / test và chặn leakage

**TODO:**
1. Giữ nguyên 10.000 test gốc; **chỉ chia 60.000 train gốc** theo tỉ lệ 90/10, `stratify`, `SEED=42` → **54.000 train + 6.000 validation**.
2. **Tạo hoặc đọc duy nhất** `artifacts/split_indices.npz` làm nguồn chia tập chung: lưu `train_idx`, `val_idx` (chỉ số của 60.000 train gốc, giữ đúng thứ tự), `test_idx` (0..9999 theo thứ tự test gốc), `seed=42` và `train_source_sha256`/`test_source_sha256` (SHA-256 của ảnh + nhãn gốc theo đúng thứ tự). Nếu đã có tệp: **đọc, kiểm hash và dùng lại, không tự chia/ghi đè**.
3. Kiểm tra shape, phân phối 10 lớp; `train_idx` và `val_idx` **không trùng, gộp đủ 60.000 chỉ số**, `test_idx` giữ đúng 10.000 ảnh test nguồn. Khi đọc lại, đối chiếu hash dữ liệu/nhãn nguồn và lấy mẫu **đúng thứ tự chỉ số lưu**, không chỉ kiểm `SEED`.
4. Tuyên bố rõ: **validation chọn mô hình/siêu tham số, test chỉ kiểm cuối**, không dùng test để tuning.

**Tái sử dụng:** `TL_Lab03.ipynb` hàm `split_and_scale`/`train_test_split`, nhưng phải sửa thành **3 tập**, không giữ `TEST_SIZE=0.3` của lab.

### 1.6. Tiền xử lý (normalization & flatten)

**TODO:**
1. Chuẩn hóa `float32` bằng pixel/255.0; kiểm min/max nằm [0,1].
2. Tạo `X_train_flat, X_val_flat, X_test_flat` có shape `(N, 784)`; giữ bản ảnh `(N,28,28)` phục vụ xem lỗi và phase CNN.
3. Nếu thêm z-score cho Logistic, chỉ **fit Standardizer trên train**, transform validation/test; lưu scaler và ghi rõ pipeline riêng của **GaussianNB (pixel/255)** và **Logistic (pixel/255, có thể thêm z-score fit train)**. Cùng mẫu/chỉ số đã khóa không có nghĩa mọi mô hình phải dùng cùng một scaler.
4. Vẽ một ảnh gốc và ảnh sau reshape về 28×28 để chứng minh không đảo trật tự pixel.

**Tái sử dụng:** `TL_Lab03.ipynb` lớp `Standardizer` (chỉ khi dùng z-score).

## 2. Gaussian Naïve Bayes tự cài đặt

### 2.1. Lý thuyết, công thức và giả định

**TODO:**
1. Viết Bayes và giả định các pixel độc lập theo lớp; lập công thức log posterior từ prior và phân phối Gaussian của từng pixel.
2. Giải thích ý nghĩa `class_prior_`, `theta_`, `var_`, `var_smoothing`; vì sao cần log-likelihood và tránh chia cho phương sai bằng 0.
3. Nêu điều kiện dữ liệu `(N,784)` và labels 0..9; nêu một ưu/nhược điểm dự kiến của giả định độc lập (chưa suy diễn kết quả).

**Tham khảo:** `NaiveBayes(1).ipynb` mục **2.1 GaussianNB** (format 6 bước).

### 2.2. Tự viết GaussianNB.fit (không dùng sklearn ở lõi)

**TODO:**
1. Tạo class `GaussianNB_NP` với `fit(X,y)`.
2. Tính priors từ tần suất lớp trong **train**, vector trung bình và phương sai cho **10 lớp × 784 pixel**.
3. Thêm smoothing/epsilon tương thích với công thức đã trình bày, kiểm shape và hữu hạn số.
4. In/thuyết minh ít nhất `classes_`, `class_prior_`, `theta_.shape`, `var_.shape`, vài tham số mẫu.

**Điều kiện hoàn thành:** không có lời gọi `sklearn.GaussianNB.fit` trong phần mô hình chính.

### 2.3. Tự viết GaussianNB.predict / log posterior

**TODO:**
1. Viết phép tính joint log-likelihood vector hóa hoặc theo batch: `log P(y=k) + sum_j log N(x_j|mu_kj,var_kj)`.
2. Trả về `predict`, có thể thêm `predict_log_proba` (normalize log probabilities nếu cần); tránh underflow, kiểm output shape `(N,)` và labels 0..9.
3. Unit test nhanh: vài ảnh train, đầu ra hữu hạn, không NaN/Inf, ảnh số 0–9.

**Sản phẩm:** Class GaussianNB tự cài hoàn chỉnh `fit/predict`.

### 2.4. Huấn luyện, tham số và kiểm chứng sklearn

**TODO:**
1. Fit **mô hình NumPy** chỉ trên train, bấm giờ fit/predict.
2. Hiển thị bảng prior 10 lớp, ảnh heatmap mean/variance của vài lớp (nếu hữu ích).
3. Fit riêng `sklearn.naive_bayes.GaussianNB` **chỉ để đối chiếu** với cùng split và smoothing tương thích; báo accuracy/F1 validation và giải thích khác biệt.
4. Nếu khảo sát `var_smoothing`, **thử và chốt bằng validation ngay tại mục 2.4**, trước khi đánh giá test ở mục 2.5; lưu cấu hình đã chọn. Tuyệt đối không chọn bằng tập test.

**Tham khảo:** `NaiveBayes(1).ipynb` mục **2.1.5** (`classes_`, `class_prior_`, `theta_`, `var_`).

### 2.5. Đánh giá GaussianNB — train / validation / test

**TODO:**
1. Dự đoán và báo cáo riêng cho cả **train, validation, test**: `accuracy`, `precision_macro`, `recall_macro`, `f1_macro`, thời gian dự đoán.
2. Vẽ **3 confusion matrix 10×10** ứng với ba tập, có nhãn 0..9, thêm bản chuẩn hóa hàng ở test (nếu tiện).
3. In precision/recall/F1 theo **từng chữ số** và chỉ ra lớp/cặp nhãn lỗi nhiều nhất.
4. Đảm bảo `macro F1` là trung bình **F1 theo từng lớp**, không phải F1 tính từ precision_macro/recall_macro.

**Tái sử dụng:** Lý thuyết macro của `BTVN_Lab01.ipynb`, plot/biện luận từ `NaiveBayes(1).ipynb` mục 2.2.6.

### 2.6. Phân tích lỗi và biện luận GaussianNB

**TODO:**
1. Liệt kê top 3 cặp nhầm lẫn `nhãn thật → nhãn dự đoán` trên validation/test, không tính đường chéo.
2. Hiển thị một số ảnh dự đoán sai có nhãn thật/nhãn dự đoán.
3. So sánh train–validation–test: underfit/overfit? Tương quan pixel ảnh hưởng NB thế nào? Nhận xét phải dựa trên số đo/ảnh vừa tạo.

## 3. Logistic Regression tuyến tính bằng Gradient Descent

### 3.1. Công thức và lựa chọn đa lớp

**TODO:**
1. Trình bày `sigmoid`, binary cross-entropy, gradient, L2 không phạt bias và phép cập nhật `w ← w - η∇J`.
2. Chọn **One-vs-Rest** để mở rộng Logistic nhị phân thành **10 mô hình**; dự đoán lớp có score/logit lớn nhất.
3. Giải thích learning rate, epoch, mini-batch, loss curve, stopping criterion và vấn đề class imbalance của OvR.

**Tái sử dụng:** `TL_Lab03.ipynb` phần **2.2 Logistic Regression**, **2.4 OneVsRest**.

### 3.2. Tái sử dụng khung NumPy từ TL_Lab03

**TODO:**
1. Mang sang các utility phù hợp: `add_bias`, `sigmoid` ổn định số học, `LogisticRegressionNP`, `OneVsRest` (hoặc cấu trúc tương đương).
2. **Hoàn thiện `LogisticRegressionNP.fit`**: file mẫu hiện có `# TODO` và `raise NotImplementedError`; tính loss/gradient/cập nhật đúng. Không được giữ stub.
3. Điều chỉnh input `(N,784)` và dữ liệu OvR nhãn {-1,+1}; bổ sung mini-batch và seed nếu full-batch 10 lớp quá chậm.
4. Viết kiểm tra gradient/giảm loss trên subset nhỏ trước khi train toàn bộ.

**Điểm tự cài đặt bắt buộc:** không gọi `sklearn.linear_model.LogisticRegression` để train mô hình chính.

### 3.3. Fit Logistic GD và theo dõi hội tụ

**TODO:**
1. Huấn luyện 10 binary classifiers trên train, có ghi cấu hình `eta`, `lambda`, `max_epochs`, `batch_size`, seed.
2. Lưu `train_loss` theo epoch và (nếu hợp lý) `validation_loss` để kiểm khả năng tổng quát; kiểm weights hữu hạn, tốc độ hội tụ.
3. Chọn checkpoint/hyperparams từ validation; đo tổng thời gian fit và dự đoán cho 3 split.
4. Xuất biểu đồ loss vs epoch; mô tả khi loss tăng/dao động.

**Tham khảo:** `TL_Lab03.ipynb` phần 2.2 và 3.3.

### 3.4. Đối chiếu với sklearn (không thay phần tự cài)

**TODO:**
1. Huấn luyện mô hình tham chiếu `LogisticRegression` + `OneVsRestClassifier` trên cùng ảnh/phép chuẩn hóa/split; quy đổi hệ số L2 phù hợp hoặc ghi rõ khi không khớp hoàn toàn.
2. So sánh NumPy vs sklearn ở **validation** về accuracy, F1 macro, loss/hội tụ, fit time, giải thích chênh lệch do solver/max_iter/regularization.
3. Không dùng kết quả sklearn để điền thay kết quả mô hình tự cài trong bảng 4 phương pháp.

**Tái sử dụng:** `TL_Lab03.ipynb` **Phần 4 — đối chiếu với scikit-learn**; không copy kết quả/loss của dataset Wine.

## 4. Thí nghiệm siêu tham số, so sánh và kết luận phase

### 4.1. Một hàm đánh giá thống nhất

**TODO:**
1. Định nghĩa `evaluate_model(model, X, y, split_name)` hoặc hàm tương tự: trả về metrics chuẩn 4 số + confusion matrix + prediction time.
2. Chuẩn hóa schema CSV: `phase, model, split, accuracy, precision_macro, recall_macro, f1_macro, train_seconds, predict_seconds, n_samples, seed, params`.
3. Kiểm `labels=range(10)`, macro average, `zero_division=0` và thứ tự split cố định.

**Tái sử dụng:** `BTVN_Lab01.ipynb` công thức; `TL_Lab03.ipynb` hàm `evaluate`, `summarize`.

### 4.2. Thí nghiệm learning rate của Logistic GD (bắt buộc trong phase)

**TODO:**
1. Chọn ít nhất 3 mức learning rate **(gợi ý `0.01, 0.05, 0.1`, điều chỉnh theo kết quả ổn định)**; giữ nguyên seed, split, L2, epochs, batch size.
2. Train mỗi cấu hình trên train; chỉ dùng **validation** để ghi accuracy/F1 macro và chọn best.
3. **Vẽ ≥ 1 biểu đồ** `learning_rate → validation F1 macro` (hoặc accuracy); ghi tiêu đề, trục, legend, nêu xu hướng từ số liệu.
4. Tùy chọn: tổng hợp biểu đồ `var_smoothing` **đã thử/chốt trên validation ở mục 2.4** (trước lần đánh giá test mục 2.5); không tuning GaussianNB sau khi xem test.

**Lưu ý:** test không dùng để chọn learning rate.

### 4.3. Đánh giá Logistic — train / validation / test

**TODO:**
1. Báo `accuracy`, `precision_macro`, `recall_macro`, `f1_macro`, `predict_seconds` trên cả **3 split**.
2. Vẽ **3 confusion matrix 10×10**, nhãn 0..9, hiển thị rõ các cặp sai nổi bật.
3. Báo metric per-digit, ít nhất vài ví dụ ảnh nhầm, kiểm overfit/underfit qua train vs validation/test.
4. Đặt kết quả cạnh GaussianNB nhưng chưa dùng test để chỉnh lại siêu tham số.

### 4.4. So sánh nội bộ và phân tích

**TODO:**
1. Lập bảng GaussianNB vs Logistic cho cả **train / validation / test**: 4 metrics + fit/predict seconds.
2. Vẽ biểu đồ so sánh F1 macro hoặc accuracy trên validation/test; lưu đồ thị.
3. Nhận xét dựa trên số liệu: giả định/giải thích NB vs trọng số tuyến tính Logistic, lỗi nổi bật, thời gian và độ khó tuning.
4. Ghi phần công thức + pseudo-code, thảo luận/hạn chế cho báo cáo.

### 4.5. Lưu kết quả & bàn giao Phase 2/3

**TODO:**
1. Lưu **`results/phase1_metrics.csv`** đủ **6 hàng** (2 mô hình × 3 split) theo schema plan; có config tốt nhất.
2. Kiểm `artifacts/split_indices.npz` **bắt buộc** đã tạo/khóa từ mục 1.5 (không thay bằng hàm chia lại), kèm hash dữ liệu nguồn theo thứ tự; lưu tham số model (nếu cần) và figures.
3. Đưa một ô tóm tắt cấu hình được chọn theo validation và checklist chạy lại notebook từ đầu.
4. Kiểm mọi kết quả xuất từ model **tự cài**, không dùng model sklearn baseline vào CSV tổng hợp.

---

## 5. Câu hỏi thảo luận và tài liệu tham khảo

**TODO — mỗi câu có bằng chứng từ bảng/hình:**
1. Vì sao GaussianNB có thể sai do pixel lân cận tương quan?
2. `eta` ảnh hưởng thế nào đến tốc độ hội tụ/dao động loss của Logistic?
3. Cặp chữ số nào dễ nhầm với mỗi mô hình? Confusion matrix chứng minh điều đó ra sao?
4. Mô hình nào thích hợp khi cần giải thích quyết định dự đoán? Vì sao?
5. Ghi nguồn 3 notebook gốc, tài liệu công thức bổ sung và khai báo AI nếu có.

**DONE:** [ ] Đủ 2 mô hình lõi tự cài; [ ] đủ metrics & CM 3 split; [ ] ≥1 hình hyperparam; [ ] CSV + ảnh + nhận xét; [ ] không leakage.